In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# QUICK-PDE: ColibriTD의 Qiskit Function
> **Note:** Qiskit Functions는 IBM Quantum&reg; Premium Plan, Flex Plan, On-Prem(IBM Quantum Platform API 경유) Plan 사용자에게 제공되는 실험적 기능입니다. 현재 미리 보기 릴리스 상태이며 변경될 수 있습니다.
## 개요
여기에 소개된 편미분 방정식(PDE) 솔버는 당사의 Quantum Innovative Computing Kit(QUICK) 플랫폼(QUICK-PDE)의 일부이며, Qiskit Function으로 패키징되어 있습니다. QUICK-PDE 함수를 사용하면 IBM Quantum QPU에서 도메인 특화 편미분 방정식을 풀 수 있습니다. 이 함수는 [ColibriTD의 H-DES 설명 논문](https://arxiv.org/abs/2410.01130)에 기술된 알고리즘을 기반으로 합니다. 이 알고리즘은 전산 유체 역학(CFD)과 재료 변형(MD)을 시작으로, 곧 추가될 다른 사용 사례까지 복잡한 다중 물리 문제를 해결할 수 있습니다.

미분 방정식을 다루기 위해, 시험 해(trial solution)는 직교 함수의 선형 결합(일반적으로 체비쇼프 다항식, 더 구체적으로는 $2^n$개이며 여기서 $n$은 함수를 인코딩하는 Qubit의 수)으로 인코딩되고, Variable Quantum Circuit(VQC)의 각도로 매개변수화됩니다. Ansatz는 함수를 인코딩하는 상태를 생성하며, 이는 모든 점에서 함수를 평가할 수 있는 관측량(observable)의 조합으로 평가됩니다. 그런 다음 미분 방정식이 인코딩된 손실 함수를 평가하고, 하이브리드 루프에서 각도를 미세 조정할 수 있습니다(아래 그림 참조). 시험 해는 만족스러운 결과에 도달할 때까지 실제 해에 점점 더 가까워집니다.

![QUICK-PDE 함수의 워크플로우](../docs/images/guides/colibritd-equation-solver/diagram.svg)

이 하이브리드 루프 외에도, 여러 최적화기를 함께 연결할 수도 있습니다. 이는 전역 최적화기로 좋은 각도 집합을 찾은 다음, 더 세밀한 최적화기로 기울기(gradient)를 따라 최적의 인접 각도 집합을 찾고자 할 때 유용합니다. 전산 유체 역학(CFD)의 경우 기본 최적화 순서가 최상의 결과를 제공하지만, 재료 변형(MD)의 경우 기본값도 좋은 결과를 제공하면서 문제별 이점을 위해 추가로 구성할 수 있습니다.

함수의 각 변수에 대해 Qubit의 수를 지정한다는 점을 참고하세요(직접 조정해볼 수 있습니다). 동일한 Circuit 10개를 쌓고 하나의 큰 Circuit 전체에서 서로 다른 Qubit에 대해 동일한 관측량 10개를 평가함으로써, 노이즈 학습기 방법에 의존하는 CMA 최적화 프로세스 내에서 노이즈를 완화하고 필요한 샷 수를 크게 줄일 수 있습니다.
## 입력/출력
### 전산 유체 역학
비점성 버거스 방정식(inviscid Burgers' equation)은 다음과 같이 비점성 유체의 흐름을 모델링합니다:

$$\frac{\partial u}{\partial t} + u\frac{\partial u}{\partial x} = 0,$$

$u$는 유체 속도 필드를 나타냅니다. 이 사용 사례에는 시간적 경계 조건이 있습니다. 초기 조건을 선택한 후 시스템이 이완되도록 할 수 있습니다. 현재 허용되는 초기 조건은 선형 함수 $ax + b$뿐입니다.

CFD 미분 방정식의 인수는 다음과 같이 고정된 격자에 있습니다:

- $t$는 0에서 0.95 사이이며 30개의 샘플 포인트를 가집니다. $x$는 0에서 0.95 사이이며 스텝 크기는 0.2375입니다.

### 재료 변형
이 사용 사례는 공간에 고정된 막대의 한쪽 끝을 당기는 1차원 인장 시험으로 준탄성 변형(hypoelastic deformation)에 초점을 맞춥니다. 문제를 다음과 같이 설명합니다:

$$u' - \frac{\sigma}{3K} - \frac{2}{\sqrt{3}}\epsilon_0\left(\frac{\sigma'}{\sigma_0\sqrt{3}}\right)^n = 0$$

$$\sigma' - b = 0,$$

$K$는 늘어나는 재료의 체적 탄성률, $n$은 멱법칙(power law)의 지수, $b$는 단위 질량당 힘, $\epsilon_0$는 비례 응력 한계, $\sigma_0$는 비례 변형률 한계, $u$는 응력 함수, $\sigma$는 변형률 함수를 나타냅니다.

고려되는 막대는 단위 길이를 가집니다. 이 사용 사례에는 표면 응력 $t$, 즉 막대를 늘리는 데 필요한 일의 양에 대한 경계 조건이 있습니다.

MD 미분 방정식의 인수는 다음과 같이 고정된 격자에 있습니다:

- $x$는 0에서 1 사이이며 스텝 크기는 0.04입니다.
### 입력
QUICK-PDE Qiskit Function을 실행하기 위해 다음 매개변수를 조정할 수 있습니다:

| 이름              | 타입                                                 | 설명                                                                                                                                                                                                                                                                     | 사용 사례 특화 | 예시                  |
| ----------------- | ---------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ----------------- | ------------------------ |
| use_case          | `Literal["MD", "CFD"]`                               | 풀 미분 방정식 시스템을 선택하는 토글                                                                                                                                                                                                                                  | 아니요                | `"CFD"`                  |
| parameters        | `dict[str, Any]`                                     | 미분 방정식의 매개변수(자세한 내용은 다음 표 참조)                                                                                                                                                                                                                              | 아니요                | `{"a": 1.0, "b": 1.0}`   |
| nb_qubits         | `Optional[dict[str, dict[str, int]]]`                | 함수 및 변수별 Qubit 수. 함수에서 최적화된 값을 선택하지만, 더 나은 조합을 찾고 싶다면 기본값을 재정의할 수 있습니다                                                                                                                                         | 아니요                | `{"u": {"x": 1, "t":3}}` |
| depth             | `Optional[dict[str, int]]`                           | 함수별 ansatz 깊이. 함수에서 최적화된 값을 선택하지만, 더 나은 조합을 찾고 싶다면 기본값을 재정의할 수 있습니다                                                                                                                                           | 아니요                | `{"u": 4}`               |
| optimizer         | `Optional[list[str]]`                                | 사용할 최적화기. [`cma` 파이썬 라이브러리](https://github.com/CMA-ES/pycma)의 "CMAES" 또는 [scipy의 최적화기](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html) 중 하나                                                                   | MD                | `"SLSQP"`                |
| shots             | `Optional[list[int]]`                                | 각 Circuit 실행에 사용되는 샷 수. 여러 최적화 단계가 필요하므로 목록의 길이는 사용되는 최적화기 수와 같아야 합니다(CFD의 경우 4개). MD의 경우 기본값은 `[50_000] * nb_optimizers`이고 CFD의 경우 `[5_00, 2_000, 5_000, 10_000]`입니다 | 아니요                | `[15_000, 30_000]`       |
| optimizer_options | `Optional[dict[str, Any]]`                           | 최적화기에 전달할 옵션. 이 입력의 세부 사항은 사용된 최적화기에 따라 다릅니다. 구체적인 내용은 사용된 최적화기의 문서를 참조하세요                                                                                                                              | MD                | `{"maxiter": 50 }`       |
| initialization    | `Optional[Literal["RANDOM", "PHYSICALLY_INFORMED"]]` | 무작위 각도로 시작할지 스마트하게 선택된 각도로 시작할지 여부. 스마트하게 선택된 각도가 100%의 경우에 작동하지 않을 수 있다는 점에 유의하세요. 기본값은 `"PHYSICALLY_INFORMED"`입니다.                                                                                                         | 아니요                | `"RANDOM"`               |
| backend_name      | `Optional[str]`                                      | 사용할 Backend 이름.                                                                                                                                                                                                              | 아니요                | `"ibm_torino"`           |
| mode              | `Optional[Literal["job", "session", "batch"]]`        | 사용할 실행 모드. 기본값은 `"job"`입니다.                                                                                                                                                                                                                                       | 아니요                | `"job"`                  |

미분 방정식의 매개변수(물리적 매개변수 및 경계 조건)는 다음 형식을 따라야 합니다:

| 사용 사례 | 키         | 값 타입 | 설명                              | 예시 |
| -------- | ----------- | ---------- | ---------------------------------------- | ------- |
| CFD      | `a`         | `float`    | $u$ 초기값의 계수 | `1.0`   |
| CFD      | `b`         | `float`    | $u$ 초기값의 오프셋      | `1.0`   |
| MD       | `t`         | `float`    | 표면 응력                           | `12.0`  |
| MD       | `K`         | `float`    | 체적 탄성률                             | `100.0` |
| MD       | `n`         | `int`      | 멱법칙                                | `4.0`   |
| MD       | `b`         | `float`    | 단위 질량당 힘                      | `10.0`  |
| MD       | `epsilon_0` | `float`    | 비례 응력 한계                | `0.1`   |
| MD       | `sigma_0`   | `float`    | 비례 변형률 한계                | `5.0`   |
### 출력
출력은 샘플 포인트 목록과 각 포인트에서의 함수 값을 포함하는 딕셔너리입니다:

In [ ]:
from numpy import array

In [ ]:
solution = {
    "functions": {
        "u": array(
            [
                [0.01, 0.1, 1],
                [0.02, 0.2, 2],
                [0.03, 0.3, 3],
                [0.04, 0.4, 4],
            ]
        ),
    },
    "samples": {
        "t": array([0.1, 0.2, 0.3, 0.4]),
        "x": array([0.5, 0.6, 0.7]),
    },
}

솔루션 배열의 형태는 변수의 샘플에 따라 달라집니다:

In [ ]:
assert len(solution["functions"]["u"].shape) == len(solution["samples"])
for col_size, samples in zip(
    solution["functions"]["u"].shape, solution["samples"].values()
):
    assert col_size == len(samples)

함수 변수의 샘플 포인트와 솔루션 배열 차원 간의 매핑은 변수 이름의 영숫자 순서로 이루어집니다. 예를 들어 변수가 `"t"`와 `"x"`인 경우, `solution["functions"]["u"]`의 한 행은 고정된 `"t"`에 대한 솔루션 값을 나타내고, 한 열은 고정된 `"x"`에 대한 솔루션 값을 나타냅니다.

다음은 특정 좌표 집합에서 함수 값을 가져오는 방법의 예시입니다:

In [ ]:
# u(t=0.2, x=0.7) == 2
assert solution["samples"]["t"][1] == 0.2
assert solution["samples"]["x"][2] == 0.7
assert solution["functions"]["u"][1, 2] == 2

## 벤치마크

다음 표는 함수의 여러 실행에 대한 통계를 보여줍니다.

| 예시                      | Qubit 수 | 초기화        | 오류     | 총 시간 (분) | 런타임 사용량 (분) |
| ---------------------------- | ---------------- | --------------------- | --------- | ---------------- | ------------------- |
| 비점성 버거스 방정식   | 50               | `PHYSICALLY_INFORMED` | $10^{-2}$ | 66               | 25                  |
| 준탄성 1D 인장 시험  | 18               | `RANDOM`              | $10^{-2}$ | 123              | 100                 |

## 시작하기

[QUICK-PDE 함수 액세스 요청 양식](https://forms.cloud.microsoft/e/3Wi9cbjQPK)을 작성하세요. 그런 다음 로컬 환경에 [계정을 저장](/guides/functions#install-qiskit-functions-catalog-client)한 경우, 다음과 같이 함수를 선택합니다:

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

quick = catalog.load("colibritd/quick-pde")

## 예시

시작하려면 다음 예시 중 하나를 시도해 보세요:

### 전산 유체 역학 (CFD)

초기 조건이 $u(0,x) = x$로 설정된 경우, 결과는 다음과 같습니다:

In [ ]:
# launch the simulation with initial conditions u(0,x) = a*x + b
job = quick.run(use_case="cfd", physical_parameters={"a": 1.0, "b": 0.0})

Qiskit Function 워크로드의 [상태](/guides/functions#check-job-status)를 확인하거나 다음과 같이 [결과](/guides/functions#retrieve-results)를 반환합니다:

In [ ]:
print(job.status())
solution = job.result()

'QUEUED'

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

_ = plt.figure()
ax = plt.axes(projection="3d")

# plot the solution using the 3d plotting capabilities of pyplot
t, x = np.meshgrid(solution["samples"]["t"], solution["samples"]["x"])
ax.plot_surface(
    t,
    x,
    solution["functions"]["u"],
    edgecolor="royalblue",
    lw=0.25,
    rstride=26,
    cstride=26,
    alpha=0.3,
)
ax.scatter(t, x, solution["functions"]["u"], marker=".")
ax.set(xlabel="t", ylabel="x", zlabel="u(t,x)")

plt.show()

<Image src="../docs/images/guides/colibritd-pde/extracted-outputs/c42aba9b-0.avif" alt="Output of the previous code cell" />

### Material Deformation

The material deformation use case requires the physical parameters of your material and the applied force, as follows:

In [ ]:
import matplotlib.pyplot as plt

# select the properties of your material
job = quick.run(
    use_case="md",
    physical_parameters={
        "t": 12.0,
        "K": 100.0,
        "n": 4.0,
        "b": 10.0,
        "epsilon_0": 0.1,
        "sigma_0": 5.0,
    },
)

# plot the result
solution = job.result()

_ = plt.figure()
stress_plot = plt.subplot(211)
plt.plot(solution["samples"]["x"], solution["functions"]["u"])
strain_plot = plt.subplot(212)
plt.plot(solution["samples"]["x"], solution["functions"]["sigma"])

plt.show()

<Image src="../docs/images/guides/colibritd-pde/extracted-outputs/a568e325-0.avif" alt="Output of the previous code cell" />

![이전 코드 셀의 출력](../docs/images/guides/colibritd-pde/extracted-outputs/c42aba9b-0.avif)

### 재료 변형
재료 변형 사용 사례는 재료의 물리적 특성과 적용된 힘을 필요로 합니다. 다음과 같이 사용합니다:

In [ ]:
job = quick.run(use_case="mdf", physical_params={})

print(job.error_message())


# or write a wrapper around it for a more human readable version
def pprint_error(job):
    print("".join(eval(job.error_message())["error"]))


print("___")
pprint_error(job)

{"error": ["qiskit.exceptions.QiskitError: 'Unknown argument \"physical_params\", did you make a typo? -- https://docs.quantum.ibm.com/errors#1804'\n"]}
___
qiskit.exceptions.QiskitError: 'Unknown argument "physical_params", did you make a typo? -- https://docs.quantum.ibm.com/errors#1804'



![이전 코드 셀의 출력](../docs/images/guides/colibritd-pde/extracted-outputs/a568e325-0.avif)

## 오류 메시지 가져오기

워크로드 상태가 `ERROR`인 경우, `job.error_message()`를 사용하여 디버깅에 도움이 되는 오류 메시지를 가져옵니다. 다음과 같이 사용합니다: